In [1]:
import os

os.path.exists(r"C:\webcrawl\crawl.warc")

True

In [2]:
from warcio.archiveiterator import ArchiveIterator

warc_file = r"C:\webcrawl\crawl.warc"

with open(warc_file, "rb") as stream:
    for record in ArchiveIterator(stream):
        print("Record type:", record.rec_type)
        print("URL:", record.rec_headers.get_header("WARC-Target-URI"))

Record type: response
URL: https://example.com/


In [3]:
with open(warc_file, "rb") as stream:
    for record in ArchiveIterator(stream):
        if record.rec_type == "response":
            content = record.content_stream().read()
            print(content.decode("utf-8"))

<!doctype html><html lang="en"><head><title>Example Domain</title><link rel="icon" href="data:,"><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>



In [5]:
from warcio.archiveiterator import ArchiveIterator
from bs4 import BeautifulSoup

warc_file = r"C:\webcrawl\crawl.warc"

with open(warc_file, "rb") as stream:
    for record in ArchiveIterator(stream):
        if record.rec_type == "response":
            content = record.content_stream().read()

            soup = BeautifulSoup(content, "html.parser")

            title = soup.title

            if title:
                print("Page Title:", title.get_text(strip=True))
            else:
                print("No title found")

Page Title: Example Domain


In [6]:
from warcio.warcwriter import WARCWriter
from warcio.statusandheaders import StatusAndHeaders
import io
import os

crawl_folder = r"C:\webcrawl\quotes\quotes.toscrape.com"
warc_file = r"C:\webcrawl\quotes\quotes.warc"

with open(warc_file, "wb") as output:
    writer = WARCWriter(output, gzip=True)

    for root, dirs, files in os.walk(crawl_folder):
        for file in files:
            file_path = os.path.join(root, file)

            # We only want HTML pages
            if file == "index.html" or "." not in file:
                with open(file_path, "rb") as f:
                    content = f.read()

                relative_path = os.path.relpath(file_path, crawl_folder)
                url = "https://quotes.toscrape.com/" + relative_path.replace("\\", "/")

                headers = StatusAndHeaders(
                    "200 OK",
                    [
                        ("Content-Type", "text/html; charset=UTF-8"),
                    ],
                    protocol="HTTP/1.1"
                )

                record = writer.create_warc_record(
                    url,
                    "response",
                    payload=io.BytesIO(content),
                    http_headers=headers
                )

                writer.write_record(record)

print(f"WARC created: {warc_file}")

WARC created: C:\webcrawl\quotes\quotes.warc


In [7]:
from warcio.archiveiterator import ArchiveIterator

warc_file = r"C:\webcrawl\quotes\quotes.warc"

count = 0

with open(warc_file, "rb") as stream:
    for record in ArchiveIterator(stream):
        if record.rec_type == "response":
            count += 1
            print("Record:", count)
            print("URL:", record.rec_headers.get_header("WARC-Target-URI"))
            print()

print("Total response records:", count)

Record: 1
URL: https://quotes.toscrape.com/index.html

Record: 2
URL: https://quotes.toscrape.com/login

Record: 3
URL: https://quotes.toscrape.com/author/Albert-Einstein

Record: 4
URL: https://quotes.toscrape.com/author/Andre-Gide

Record: 5
URL: https://quotes.toscrape.com/author/Eleanor-Roosevelt

Record: 6
URL: https://quotes.toscrape.com/author/J-K-Rowling

Record: 7
URL: https://quotes.toscrape.com/author/Jane-Austen

Record: 8
URL: https://quotes.toscrape.com/author/Marilyn-Monroe

Record: 9
URL: https://quotes.toscrape.com/author/Steve-Martin

Record: 10
URL: https://quotes.toscrape.com/author/Thomas-A-Edison

Record: 11
URL: https://quotes.toscrape.com/page/2/index.html

Record: 12
URL: https://quotes.toscrape.com/tag/abilities/page/1/index.html

Record: 13
URL: https://quotes.toscrape.com/tag/adulthood/page/1/index.html

Record: 14
URL: https://quotes.toscrape.com/tag/aliteracy/page/1/index.html

Record: 15
URL: https://quotes.toscrape.com/tag/be-yourself/page/1/index.html



In [8]:
from warcio.archiveiterator import ArchiveIterator
from bs4 import BeautifulSoup

warc_file = r"C:\webcrawl\quotes\quotes.warc"

with open(warc_file, "rb") as stream:
    for record in ArchiveIterator(stream):

        if record.rec_type == "response":

            content = record.content_stream().read()

            soup = BeautifulSoup(content, "html.parser")

            title = soup.title

            if title:
                print("URL:", record.rec_headers.get_header("WARC-Target-URI"))
                print("Title:", title.get_text(strip=True))
                print("-" * 60)

URL: https://quotes.toscrape.com/index.html
Title: Quotes to Scrape
------------------------------------------------------------
URL: https://quotes.toscrape.com/login
Title: Quotes to Scrape
------------------------------------------------------------
URL: https://quotes.toscrape.com/author/Albert-Einstein
Title: Quotes to Scrape
------------------------------------------------------------
URL: https://quotes.toscrape.com/author/Andre-Gide
Title: Quotes to Scrape
------------------------------------------------------------
URL: https://quotes.toscrape.com/author/Eleanor-Roosevelt
Title: Quotes to Scrape
------------------------------------------------------------
URL: https://quotes.toscrape.com/author/J-K-Rowling
Title: Quotes to Scrape
------------------------------------------------------------
URL: https://quotes.toscrape.com/author/Jane-Austen
Title: Quotes to Scrape
------------------------------------------------------------
URL: https://quotes.toscrape.com/author/Marilyn-Monro